# Naturalistic EMA prediction based on ACC - Tremor use case

## 0) Imports

- document versions for reproducibility

In [ ]:
# import packages
import datetime as dt
import pandas as pd
import numpy as np
import os
import sys
import csv
import json
import importlib
from itertools import product, compress
import matplotlib.pyplot as plt
from scipy.stats import pearsonr, variation

from dataclasses import dataclass, field


In [ ]:
print('Python sys', sys.version)
print('pandas', pd.__version__)
print('numpy', np.__version__)
# print('mne_bids', mne_bids.__version__)
# print('mne', mne.__version__)
# print('sci-py', scipy.__version__)
# print('sci-kit learn', sk.__version__)
# print('matplotlib', plt_version)

"""
Python sys 3.11.5 | packaged by Anaconda, Inc. | (main, Sep 11 2023, 13:26:23) [MSC v.1916 64 bit (AMD64)]
pandas 2.1.1
numpy 1.26.0

from 16.09

Python sys 3.11.5 | packaged by Anaconda, Inc. | (main, Sep 11 2023, 13:26:23) [MSC v.1916 64 bit (AMD64)]
pandas 2.3.2
numpy 2.3.3
"""

Import custom functions

In [ ]:
# from dbs_home repo
from dbs_home.load_raw.main_load_raw import loadSubject 
import dbs_home.utils.helpers as home_helpers
import dbs_home.utils.ema_utils as home_ema_utils
import dbs_home.plot_data.plot_compliance as plot_home_compl
import dbs_home.preprocessing.preparing_ema as home_ema_prep

from dbs_home.preprocessing import get_submovements
import dbs_home.preprocessing.acc_preprocessing as acc_prep
import dbs_home.preprocessing.submovement_processing as submove_proc
import dbs_home.load_raw.load_watch_raw as load_watch


In [ ]:
# from current repo
from utils import load_utils, load_data, prep_data
import utils.acc_features as acc_fts
import utils.ft_extract_class as ft_extr_class
import utils.feat_extraction as ft_extr
import utils.data_handling_ema_acc as data_handling

from plotting import plot_help



## 1) Load and check naturalistic data

In [ ]:
# set paths

feas_data_path = os.path.join(
    os.path.dirname(load_utils.get_onedrive_path()),
    'PROJECTS', 'home_feasibility'
)
feas_fig_path = os.path.join(
    load_utils.get_onedrive_path('figures'),
    'feasibility'
)



#### Load ACC data
- create SVM
- filter data within the dataclass
- potential inclusions:
    - hm20, ses01
    - hm22, ses01+ses02

In [ ]:
# tremor
sub_id = 'hm22'  
ses_id = 'ses03'

FT_PARAMS_VERSION = 'v4'  # v4 tremor version

In [ ]:
# import naturalistic data via dbs_home repo




### test days for hm22-ses01  # dyskinesia
# dev_day_selection = ['2025-07-17', '2025-07-18']
# dev_day_selection = [f'2025-07-{d}' for d in np.arange(17, 31)]
dev_day_selection = []

### test days for hm20-ses01  # tremor
# dev_day_selection = [
#     '2025-06-13', '2025-06-14',
#     '2025-06-15', '2025-06-16'
# ]

home_dat = loadSubject(
    sub=sub_id,
    ses=ses_id,
    incl_STEPS=False,
    incl_EPHYS=False,
    incl_EMA=True,
    incl_ACC=True,
    day_selection=dev_day_selection
)

Check available EMAs

In [ ]:
plot_home_compl.plot_EMA_completion_perSession(home_dat)

## 2) ACC processing incl. Feature extraction

#### 2a) Extract features from Acc-Windows aligned to EMAs

In [ ]:
importlib.reload(load_watch)
importlib.reload(ft_extr_class)
importlib.reload(acc_fts)
importlib.reload(data_handling)
importlib.reload(ft_extr)


xy_dict = {}


ft_settings = {
    'nosm_allwindow': {
        'EXTRACT_FT_FROM_SMs': False,
        'EXTRACT_FT_FULL_WIN': True,
        'ACC_FEATS_on_SINGLE_MOVES': True
    },
    'sm_merged': {
        'EXTRACT_FT_FROM_SMs': True,
        'EXTRACT_FT_FULL_WIN': False,
        'ACC_FEATS_on_SINGLE_MOVES': False
    },
    'sm_single': {
        'EXTRACT_FT_FROM_SMs': True,
        'EXTRACT_FT_FULL_WIN': False,
        'ACC_FEATS_on_SINGLE_MOVES': True
    }
}




for ft_type, ft_settings in ft_settings.items():

    xy_dict[ft_type] = ft_extr.get_features_per_session(
        home_dat=home_dat,
        sub_id=sub_id,
        ses_id=ses_id,
        FT_PARAMS_VERSION=FT_PARAMS_VERSION,
        LOAD_SAVE_FEATS = True,
        # define how features should be extracted
        STORE_SUBMOVES = True,
        # plotting settings
        SAVE_PLOT = False,
        SHOW_PLOT = False,
        verbose=True,
        **ft_settings,
    )

In [ ]:
[f'{k}: {v.shape}, column-names: {list(v.keys())}' for k, v in xy_dict.items()]

#### 2b) feature extractions for full days

updated function to directly get feature dataframes, without home_dat requirement

In [ ]:
importlib.reload(acc_prep)
importlib.reload(load_watch)
importlib.reload(ft_extr)


In [ ]:
importlib.reload(ft_extr)

dfs_sessions = {}

sub_id = 'hm22'

for ses_id in ['ses01', 'ses03']:
    
    tempdf = ft_extr.get_feat_df_for_pred(
        sub_id=sub_id, ses_id=ses_id,
        ft_set_sel = 'sm_single',
        ONLY_EMA_WINDOWS=False,
    )
    dfs_sessions[ses_id] = tempdf


## 3a) Evaluate extracted Features

- Hssayeni et al, Scientific Reports 2021
    - strongest wrist-features: angular velocity, standard deviation, power of secondary frequency, power of 1â€“4 Hz band, and Shannon Entropy (r = 0.82  - r = 0.75)

- from svm: classic features
- include cross-corr between pc1 and pc2


In [ ]:
from plotting import plot_home_preds as plt_preds
import plotting.plot_ft_explor as plt_fts

define selected data

In [ ]:
sub_id = 'hm22'
# SES = 'ses03'
FT_TYPE = 'sm_single'    # nosm_allwindow, sm_merged, sm_single
FT_PARAMS_VERSION = 'v4'  # v4 tremor version
EMA_ref = 'tremor'



In [ ]:
importlib.reload(ft_extr)


xy_dict = {}

for ses_id in ['ses01', 'ses02', 'ses03']:

    # if ses_id != SES: continue
    if ses_id == 'ses02': continue

    tempdf = ft_extr.get_feat_df_for_pred(
        sub_id=sub_id,
        ses_id=ses_id,
        ft_set_sel=FT_TYPE,
        FT_PARAMS_VERSION=FT_PARAMS_VERSION,
        ONLY_EMA_WINDOWS=True,
        # **ft_settings[FT_TYPE]
    )
    xy_dict[ses_id] = tempdf

In [ ]:
# ft_df = xy_dict[SES].copy()
ft_df = pd.concat(xy_dict.values()).reset_index(drop=True)

check selected variables in dataframe and first rows of dataframe

In [ ]:
print(ft_df.shape)
print(ft_df.keys())
print(ft_df.head())
print([f'tremor "{k}": {v} counts' for k, v in zip(
    *np.unique(ft_df[EMA_ref], return_counts=True))])


In [ ]:
importlib.reload(ft_extr)

# define which features to include in prediction and which to exclude (times, ema)
feats_incl, EMA_CODING = ft_extr.get_feat_params(FT_PARAMS_VERSION)
keys_excl = ['timestamp'] + list(EMA_CODING.keys())  # times + ema

Visualize feature distributions

In [ ]:
importlib.reload(plt_fts)

save_plot = False


for ft_name in ft_df.keys():
    print(f'\n{ft_name}')
    # if ft_name not in ft_df.keys():
    #     print(f'Feature {ft_name} not found in dataframe columns. Skipping.')
    #     continue
    if ft_name in keys_excl:
        print(f'Feature {ft_name} is not included in the feature set. Skipping.')
        continue
    
    ### plot distributions with histo, scatter, and boxes
    plt_fts.plot_ft_distribution(
        ft_df=ft_df,
        ft_name=ft_name,
        EMA_ref=EMA_ref,
        sub_id=sub_id,
        SES=SES,
        FT_TYPE=FT_TYPE,
        FT_PARAMS_VERSION=FT_PARAMS_VERSION,
        save_plot=save_plot,
    )



## 3b) Cluster wearable-features and validate with EMA scores



In [ ]:
import utils.pred_utils as pred_utils
import utils.pred_clustering as clust

from scipy.stats import mannwhitneyu
import utils.stats as utils_stats

In [ ]:
# define window for clustering
cv_df = xy_dict['ses01'].copy()
cv_df = pd.concat([cv_df, xy_dict['ses03']]).reset_index(drop=True) 

print(f'feature type used: {FT_TYPE}')
print(cv_df.keys(), cv_df.shape)

FT_PARAMS_VERSION = 'v4'  # v4 first tremor version
EMA_Y = 'tremor'

EXCL_HR = True  # exclude heart rate features (not available in all timepoints)
SKEW_THRESH = 1  # None is no log-transform, otherwise log-transform features with skew above threshold

OUTLIER_REMOVAL = 'zscore'  # 'none', 'zscore', 'isoforest'
USE_PCA=True
N_CLUST = 3
TRANSFORM_Y = None
# TRANSFORM_Y = {1: [1, 2], 2: [3, 4, 5, 6], 3: [7, 8, 9]}  # transform LID into 3 classes: 1, 2 (1-4), 3 (>4)

PRED_FTS = pred_utils.get_keys_incl(cv_df.keys(), excl_hr=EXCL_HR,)
print(f'model trained on features: {PRED_FTS}')


In [ ]:
importlib.reload(pred_utils)
importlib.reload(clust)

# compare both methods, pick one for permutation testing

SEL_CLUST_METHOD_TESTING = 'kmeans'

n_perm = 0
rho_perm = []

### define X, y
X = cv_df[PRED_FTS].values.copy()
y = cv_df[EMA_Y].values.copy().astype(float)

true_clust_results = clust.run_clustering(
    X=X, y=y, transform_y=TRANSFORM_Y,
    n_clusters=N_CLUST, cluster_method='both',
    USE_PCA=USE_PCA, n_comp=6, plot_pca=True, skew_thresh=SKEW_THRESH,
    outlier_removal=OUTLIER_REMOVAL, verbose=True,
    return_fitted_params=True,
)
rho_true = true_clust_results[SEL_CLUST_METHOD_TESTING]['spearman_rho']



np.random.seed(27)

for i in range(n_perm):

    X = cv_df[PRED_FTS].values.copy()
    y = cv_df[EMA_Y].values.copy().astype(float)
    np.random.shuffle(y)

    clust_results = clust.run_clustering(
        X=X, y=y, n_clusters=N_CLUST, cluster_method=SEL_CLUST_METHOD_TESTING,
        USE_PCA=USE_PCA, n_comp=6, plot_pca=False, skew_thresh=SKEW_THRESH,
        outlier_removal=OUTLIER_REMOVAL, verbose=False,
    )
    rho_perm.append(clust_results[SEL_CLUST_METHOD_TESTING]['spearman_rho'])

    # print(f'\npermutation done {i+1}/{n_perm}')
    

Tested trained clusters on all-window data

In [ ]:
### EXTRACT TEST DATA 

test_sess_ids, test_timestamps = [], []  # store to which session test-samples belong

for i, ses_id in enumerate(['ses01', 'ses03', 'ses02']):
    
    tempdf = ft_extr.get_feat_df_for_pred(
        sub_id=sub_id,
        ses_id=ses_id,
        ft_set_sel=FT_TYPE,
        FT_PARAMS_VERSION=FT_PARAMS_VERSION,
        ONLY_EMA_WINDOWS=False,
    )
    
    if i == 0: test_df = tempdf.copy()
    else: test_df = pd.concat([test_df, tempdf],).reset_index(drop=True)
    test_sess_ids.extend([ses_id] * len(tempdf))
    test_timestamps.extend(list(tempdf.index.values))

print(f'test_df shape: {test_df.shape}\n\ntestdf columns: {test_df.keys()}')

X = test_df[PRED_FTS].values.copy()
print(f'X shape: {X.shape}\n\nincl columns in X: {PRED_FTS}')


In [ ]:
importlib.reload(pred_utils)
importlib.reload(clust)


test_clust_results = clust.run_clustering(
    X=X,
    return_nanrow_bool=True,
    apply_fitted_skewed_fts=true_clust_results['fitted_model']['skewed_ft_bool'],
    apply_fitted_pca=true_clust_results['fitted_model']['pca'],
    apply_fitted_kmeans=true_clust_results['fitted_model']['fitted_km'],
    return_fitted_params=False,
    transform_y=TRANSFORM_Y,
    n_clusters=N_CLUST,
    cluster_method='kmeans',
    USE_PCA=USE_PCA,
    n_comp=6,
    plot_pca=False,
    skew_thresh=SKEW_THRESH,
    outlier_removal=OUTLIER_REMOVAL,
    verbose=True,
)

In [ ]:
### calculate statistical significance of differences between EMA-values per labeled cluster
# n_clusters = 3, therefore three times a 1 vs 1 comparison between clusters, with bonferroni correction p<0.05/3

test_clust_results

create day-time related variables to apply as random features in LMM

In [ ]:

# get string code for daytime moment of samples
dayblock_list = [prep_data.get_daytime_block_from_str(t,) for t in test_timestamps]

# plt.plot(dayblock_list)

In [ ]:
# get numerical code indexing unique days of samples
avail_days = np.unique([prep_data.get_day_from_str(t) for t in test_timestamps])
day_code_dict = {d: i for i, d in enumerate(avail_days)}
day_list = [day_code_dict[prep_data.get_day_from_str(t)] for t in test_timestamps]

# plt.plot(day_list)

In [ ]:
### evaluated clustered EMA scores for three sessions
importlib.reload(prep_data)


# process clusters into expected EMA scores (from training fitting)
test_pred_clusters = test_clust_results['kmeans']['clust_labels']
fitted_cluster_meanEMA = true_clust_results['fitted_model']['cluster_scores']
test_pred_EMA = np.array([fitted_cluster_meanEMA[c] for c in test_pred_clusters])

# identify originating sessions to samples 
test_sessions = np.array(test_sess_ids)[~test_clust_results['kmeans']['nan_row_bool']]

# create random variables for LMM based on day and daytime
test_stamps = np.array(test_timestamps)[~test_clust_results['kmeans']['nan_row_bool']]
dayblock_arr = np.array(
    [prep_data.get_daytime_block_from_str(t, RETURN_NUM=True)
     for t in test_stamps]
)
avail_days = np.unique([prep_data.get_day_from_str(t) for t in test_stamps])
day_code_dict = {d: i for i, d in enumerate(avail_days)}
daycode_arr = np.array([day_code_dict[prep_data.get_day_from_str(t)] for t in test_stamps])

# compare predicted EMA scores per originating session
for temp_ses in np.unique(test_sessions):

    temp_scores = test_pred_EMA[test_sessions == temp_ses]
    print(temp_scores.shape)
    print(f"for {temp_ses} (n={len(temp_scores)}): mean {np.mean(temp_scores):.1f}, "
          f"sd: {np.std(temp_scores):.1f}")


In [ ]:
importlib.reload(utils_stats)

# get unique ids in test data
avail_sess = np.unique(test_sessions)

for ses_A, ses_B in product(avail_sess, repeat=2):
    if ses_A >= ses_B:  # only compare each pair once and skip self-comparison
        continue

    print(f"\n\n\n{'#'*10} START STAT COMPARISON between {ses_A} and {ses_B} {'#'*10}\n")

    preds_A = test_pred_EMA[test_sessions == ses_A]
    preds_B = test_pred_EMA[test_sessions == ses_B]
    
    # perform Mann-Whitney U test
    stat, p_value = mannwhitneyu(preds_A, preds_B, alternative='two-sided')
    
    print(f'Mann-Whitney U test between {ses_A} and {ses_B}: U={stat}, p={p_value:.4f}')

    ## LMM for session combo
    depend_var = np.concatenate([preds_A, preds_B])
    indep_var = np.concatenate([np.zeros(len(preds_A)), np.ones(len(preds_B))])
    # include daytime block as independent variable
    daytime_var = np.concatenate([dayblock_arr[test_sessions == ses_A],
                                  dayblock_arr[test_sessions == ses_B]])
    indep_var = np.column_stack((indep_var, daytime_var))  # combine session and daytime into independent variables

    
    # add random_var based on unique DAY
    groups = np.concatenate([daycode_arr[test_sessions == ses_A],
                             daycode_arr[test_sessions == ses_B]])

    fix_eff, p_lmm = utils_stats.run_mixEff_wGroups(
        dep_var=depend_var,
        indep_var=indep_var,
        groups=groups,
        PRINT_RESULTS=True,
    )

## consider to premute the U-values for significance

In [ ]:
importlib.reload(plt_preds)

plt_preds.plot_session_boxes2(test_pred_EMA, test_sessions,)

deprecated

In [ ]:
### plot true score vs permuted distribution as histogram with vertical line for true score
plt.figure(figsize=(4, 4))
plt.hist(rho_perm, bins=30, alpha=0.7, color='skyblue', edgecolor='k')
plt.axvline(rho_true, color='red', linestyle='--', label=f'True rho: {rho_true:.2f}')
plt.xlabel('Spearman rho (true vs predicted)')
plt.ylabel('Frequency')
plt.title('Permutation Test for Spearman rho')
plt.legend()
plt.tight_layout()
plt.show()

p_rho_perm = (np.sum(np.array(rho_perm) >= rho_true) + 1) / (n_perm + 1)
print(f'Permutation p-value for Spearman rho: {p_rho_perm:.4f}')


In [ ]:


true_clust_results['kmeans']